In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt

from helpers import load_csv_data, create_csv_submission

%matplotlib inline

DATA_PATH = "dataset"
CACHE_PATH = os.path.join(DATA_PATH, "data.npz")

# Parsing the CSVs is slow, so cache the arrays after the first load
if os.path.exists(CACHE_PATH):
    data = np.load(CACHE_PATH)
    x_train, x_test, y_train, train_ids, test_ids = (
        data[k] for k in ["x_train", "x_test", "y_train", "train_ids", "test_ids"]
    )
else:
    x_train, x_test, y_train, train_ids, test_ids = load_csv_data(DATA_PATH)
    np.savez(
        CACHE_PATH,
        x_train=x_train,
        x_test=x_test,
        y_train=y_train,
        train_ids=train_ids,
        test_ids=test_ids,
    )

# Logistic regression expects labels in {0, 1}; the dataset uses {-1, 1}
y_train = (y_train == 1).astype(float)

print("x_train:", x_train.shape, " y_train:", y_train.shape, " x_test:", x_test.shape)
print("positive rate:", y_train.mean())
print("share of missing values in x_train:", np.isnan(x_train).mean())

In [ ]:
from preprocessing import load_feature_names, fit_preprocessing, transform

names = load_feature_names(DATA_PATH)

# All statistics come from the training set only, then are applied to both sets
params = fit_preprocessing(x_train, names, max_missing=0.8, max_categories=10, one_hot=True)
tx_train = transform(x_train, names, params)  # first column is the bias (ones)
tx_test = transform(x_test, names, params)

print("tx_train:", tx_train.shape, " tx_test:", tx_test.shape)
print("original columns kept:", len(params["columns"]), "of", len(names))
print("one-hot encoded columns:", len(params["categories"]))
print("missing values left:", np.isnan(tx_train).sum(), np.isnan(tx_test).sum())

## Sanity check: regularized logistic regression on a validation split

The preprocessing is fitted on the training part only, so the validation score is honest. Since only ~9% of labels are positive, accuracy is misleading (always predicting "no" gives 91%), so we report F1 and pick the decision threshold that maximizes it.

In [ ]:
from implementations import reg_logistic_regression, sigmoid


def f1_score(y_true, y_pred):
    """F1 score for labels in {0, 1}."""
    tp = np.sum((y_pred == 1) & (y_true == 1))
    return 2 * tp / (np.sum(y_pred == 1) + np.sum(y_true == 1))


# Hold out 20% of the training data for validation
rng = np.random.default_rng(1)
idx = rng.permutation(len(y_train))
va, tr = idx[: len(idx) // 5], idx[len(idx) // 5 :]

p = fit_preprocessing(x_train[tr], names)
tx_tr, tx_va = transform(x_train[tr], names, p), transform(x_train[va], names, p)

w, loss = reg_logistic_regression(
    y_train[tr], tx_tr, 1e-4, np.zeros(tx_tr.shape[1]), max_iters=500, gamma=0.5
)
prob = sigmoid(tx_va @ w)

thresholds = np.arange(0.05, 0.6, 0.01)
f1s = [f1_score(y_train[va], (prob >= t).astype(float)) for t in thresholds]
best = int(np.argmax(f1s))
print(f"train loss: {loss:.4f}")
print(f"validation F1: {f1s[best]:.3f} at threshold {thresholds[best]:.2f}")
print(f"validation F1 at threshold 0.5: {f1_score(y_train[va], (prob >= 0.5).astype(float)):.3f}")

plt.plot(thresholds, f1s)
plt.xlabel("decision threshold")
plt.ylabel("validation F1")
plt.show()